# The constructor families and the constraint set

*A learning exercise performed in role: a simulated mandate with no client and no institution. Nothing in this notebook is investment advice, a recommendation, or a client communication.*

**Entry point** `python3 -m portfolio_workbench.construct.families`

**Modules covered** `construct/families.py`, `construct/constraints.py`, `construct/means.py`

_Generated from the code by `python3 -m reporting.notebooks`: the module headers below are read out of the modules themselves, and the run is the entry point's own output._

## 1. What this module does, and the source of every method in it

The construction layer turns a covariance, a mean input and a constraint set into a weight vector, and does it for seven families plus their variants: equal weight, the policy book, mean-variance (sample, shrunk, Black-Litterman and no-mean), minimum variance, maximum diversification, equal risk contribution, hierarchical risk parity and mean-CVaR. The constraint set is the same one for every cell, so a difference between cells is the objective rather than the rules.

**Sources.** Every public function of the modules this notebook covers, and what it traces to. The map is checked over the code by the acceptance fixture, so a method added without a source fails a command rather than going unnoticed.

- `construct/families.py::diversification_ratio` traces to the constructor families: constrained mean-variance (Markowitz 1952), its analytic relatives and the risk-based family, each under the mandate's own constraint set
- `construct/families.py::equal_weight` traces to the constructor families: constrained mean-variance (Markowitz 1952), its analytic relatives and the risk-based family, each under the mandate's own constraint set
- `construct/families.py::erc` traces to equally weighted risk contributions, Maillard, Roncalli & Teiletche (2010), Journal of Portfolio Management 36(4)
- `construct/families.py::expected_shortfall` traces to the constructor families: constrained mean-variance (Markowitz 1952), its analytic relatives and the risk-based family, each under the mandate's own constraint set
- `construct/families.py::hierarchical_risk_parity` traces to hierarchical risk parity, Lopez de Prado (2016), Journal of Portfolio Management 42(4): single-linkage clustering on correlation distance, quasi-diagonalisation, then recursive bisection
- `construct/families.py::maximum_diversification` traces to the most diversified portfolio of Choueifaty & Coignard (2008), Journal of Portfolio Management 35(1): maximise the diversification ratio, the weighted average of the sleeves' volatilities over the portfolio's
- `construct/families.py::mean_cvar` traces to the mean-CVaR programme of Rockafellar & Uryasev (2000), Journal of Risk 2(3), as the linear programme that measures the tail of its own window
- `construct/families.py::mean_variance` traces to constrained mean-variance, Markowitz (1952), Journal of Finance 7(1), at the stated risk-aversion weight
- `construct/families.py::minimum_variance` traces to minimum variance, whose closed form against reciprocal variances is what the hand-checked identity case uses (Clarke, de Silva & Thorley 2006, Journal of Portfolio Management 33(1))
- `construct/families.py::policy` traces to the constructor families: constrained mean-variance (Markowitz 1952), its analytic relatives and the risk-based family, each under the mandate's own constraint set
- `construct/constraints.py::banded` traces to the no-trade band of the prototype's amended transition policy: a 1% absolute band per sleeve, with the residual it leaves closed by funding rather than left outside the sleeves
- `construct/constraints.py::bounded_simplex` traces to the mandate's constraint set and trading conventions of this effort: long-only, fully invested, a 35% per-sleeve cap, a 1% absolute no-trade band and a 5% one-way turnover cap on traded notional
- `construct/constraints.py::cost` traces to the mandate's cost convention: a per-side rate charged on traded notional, so the charge is twice the one-way turnover
- `construct/constraints.py::establishment` traces to the establishment cost of this effort: the whole book traded once outside the window, charged at the same rate and reported as its own line rather than amortised
- `construct/constraints.py::labelled` traces to the mandate's constraint set and trading conventions of this effort: long-only, fully invested, a 35% per-sleeve cap, a 1% absolute no-trade band and a 5% one-way turnover cap on traded notional
- `construct/constraints.py::rebalance` traces to the mandate's constraint set and trading conventions of this effort: long-only, fully invested, a 35% per-sleeve cap, a 1% absolute no-trade band and a 5% one-way turnover cap on traded notional
- `construct/constraints.py::trade` traces to the mandate's constraint set and trading conventions of this effort: long-only, fully invested, a 35% per-sleeve cap, a 1% absolute no-trade band and a 5% one-way turnover cap on traded notional
- `construct/means.py::black_litterman` traces to the Black-Litterman posterior, Black & Litterman (1992), Financial Analysts Journal 48(5), in the form Walters (2011) derives
- `construct/means.py::jorion` traces to Bayes-Stein shrinkage of the sample mean, Jorion (1986), Journal of Financial and Quantitative Analysis 21(3), DOI 10.2307/2331042
- `construct/means.py::none` traces to the no-mean cell: the estimator is dropped and the constraint set decides the book, which is the control that separates a mean input's contribution from the solver's
- `construct/means.py::posterior` traces to the mean-input axis: the sample mean, Bayes-Stein shrinkage and the Black-Litterman posterior, declared as an axis because the mean is the input the literature agrees carries the most error
- `construct/means.py::sample` traces to the mean-input axis: the sample mean, Bayes-Stein shrinkage and the Black-Litterman posterior, declared as an axis because the mean is the input the literature agrees carries the most error

## 2. Why it works this way, including what was rejected

_The module headers, verbatim: each records why the module is shaped the way it is, what was rejected, and the measurement that settled it. They are quoted here rather than restated, so the notebook cannot drift from the code._

**`construct/families.py`**

The constructor families, behind one interface.

Each family takes a window's covariance, a window's mean vector and the window's returns, and
returns one labelled target weight vector under the package's constraint set. Every family is called
with the same three inputs (a family that ignores the mean ignores it rather than being called
differently) because the comparison varies one axis at a time and a second calling convention would
be a second axis.

Five things in here are decisions rather than mechanics.

**The risk-aversion weight is a stated convention, not a preference estimate.** Mean-variance needs
the coefficient trading expected return against variance, and practitioners are documented as unable
to state it: the reading behind this file records Northfield's note that most investors cannot
numerically express their mean-variance trade-off parameter, and its rule of thumb that the
parameter in the `mean - lambda * variance` form is "typically about one sixth". This file writes
the objective as `mu'w - (gamma/2) w'Sw`, where `gamma = 2 * lambda`, so the rule of thumb is the
constant below. It is held fixed across every mean-variance cell, so the mean input is the only
thing that moves between them.

**The mean arrives already formed.** Shrinking and viewing are the mean axis and live in their own
module; the constructor receives a vector. That is what makes the axis an input choice rather than a
code path, and it is why no family here knows which input produced its mean.

**ERC is one objective, run twice.** The bounded form is the same problem with the package's own
per-sleeve cap as the bound, so the difference between the two cells is the constraint and not the
solver. Both minimise the same sum of squared deviations of the sleeve risk contributions from their
common target, which is the definition of the portfolio rather than a proxy for it: a solution whose
contributions are unequal has not solved the problem, and the spread is reported so a caller can see
how far from it the answer sits.

**Tail construction is a linear program against the window's own tail.** The Rockafellar-Uryasev
formulation minimises a level plus the mean excess beyond it, which makes expected shortfall linear
in the weights, so the cell is solved exactly rather than by iterating on a quantile. Over a
60-month window at the five percent level the tail is three observations, so the cell measures the
tails of its own window rather than a tail property of the market; that is reported rather than
smoothed over.

**A rule-based family is projected onto the constraint set, and the projection is reported.**
Equal weight and the policy portfolio sit inside the cap by construction. Hierarchical risk parity
is a bisection with no bounds to set, so it is projected, and the projection changes it whenever a
cap binds, which the report prints, because a cell that stops being the method it is named after
is a result about the constraint set.

**`construct/constraints.py`**

The fixed constraint set, and the two rules that turn a target into a trade.

Every constructor in the package returns a **target** weight vector under the same constraint set:
long-only, fully invested, and no sleeve above the per-sleeve cap. The set is declared here, beside
the rules that enforce it, rather than in the universe module: a constraint parameterises a
construction and not a universe, and a bound held away from the code that applies it is a bound
that drifts from the code that assumes it.

Four things in here have a plausible wrong version, and each is stated because this package's cost
and turnover numbers are read off them.

**A cap is not a clip.** Clipping weights at the cap and renormalising is the obvious way to enforce
it and it is wrong twice over: the renormalisation can push a sleeve back over the cap, and it
leaves the vector summing to something other than one by whatever the clip removed. The projection
here solves the constrained problem instead, by clipping and redistributing the excess among the sleeves
still below the cap, and repeats until nothing is left to give. The feasible set is empty when the
cap times the sleeve count is below one, and that is refused rather than returned as a vector that
cannot exist.

**The no-trade band bounds drift, not trading.** A sleeve is reverted to its held weight when its
target has moved less than the band, which stops the book churning on estimation noise. It does
**not** bound turnover: eleven sleeves each drifting just under a one-percent band can still trade
several percent in a month, and on this panel the minimum-variance cell traded more than twice the
turnover cap with the band in place. The cap therefore needs its own rule, and both are applied
here, band first.

**The turnover cap scales the trade vector.** The alternative (constrain the optimiser against the
current weights) changes each constructor's objective and so changes the method under comparison,
which is exactly what the fixed constraint set exists to prevent. Scaling lands the book on a convex
combination of where it is and where the banded target sits; both ends are feasible, so the point
between them is, and the scaling is reported as binding rather than applied quietly. The trade it
gives up is not lost: the same target is re-tested next month against the same cap.

**Cost is charged on traded notional, which is twice the one-way turnover.** The per-side rate
applies to what is actually traded, and one-way turnover counts only the buys (or only the sells) of
a book that stays fully invested, so it is half the traded notional. Writing the rate beside the
one-way figure instead halves every cost in the project, which is the error this module states the
convention to prevent.

The establishment trade is not a measured rebalance and neither rule applies to it. Every cell is
funded from cash at its own first target, which trades the whole book once; that cost is reported
as its own line, outside the measured window and outside the cap, because a rule that cannot be
satisfied at inception is not a rule.

**`construct/means.py`**

The mean-input axis: four answers to how far the estimated mean can be trusted, behind one
interface.

Every input returns the same thing (one mean vector over the sleeves, plus the diagnostics that say
how it was formed) and the constructor that consumes it never learns which input produced it. That
is what makes the axis an input choice rather than a code path, and it is why the four cells can be
compared with the optimiser held fixed.

**Shrinking estimates its own intensity.** The Bayes-Stein estimator pulls the sample mean toward the
minimum-variance portfolio's implied mean, and the intensity is not a parameter of this module: it is
estimated from the window, and it is returned so it can be printed per window. A reader expecting a
fixed shrinkage fraction will find a small one, and the small value is the window's own statement
that its sample mean, measured against its own dispersion, is not far enough from the target to be
worth shrinking hard. Nothing is chosen here, so nothing here can be tuned to flatter the cell.

**The view is stated, and so is its uncertainty.** Black-Litterman needs a prior, a view and the
view's uncertainty, and a cell that leaves any of the three implicit is reporting a number whose
content is unknown. The prior is the returns implied by the mandate's own policy holdings (the
reverse optimisation that defines the equilibrium in the method's own construction) at the same
risk-aversion weight mean-variance uses. The view is the window's sample mean, one view per sleeve.
The uncertainty is the sampling variance of that estimate, one over the observation count times the
sleeve's own variance, which states the belief in the sample mean only as far as its own standard
error reaches, rather than naming a chosen confidence.

**What the tau-over-omega ratio does and does not move.** With the prior's scaling set to one over
the observation count and the view's uncertainty to the same factor, the observation count cancels
and the posterior depends only on the ratio between how much the prior is trusted and how much the
view is. So a rerun at a different sample size is the same answer, and the sensitivity that matters
is the ratio, which is why the cell reports the base case and the ratio run beside it instead of a
single number whose provenance a reader cannot reconstruct.

**All four are sample-based, and the sample is the window.** Every input is estimated from the same
trailing window the covariance comes from, so the axis carries no look-ahead of its own; a mean taken
from somewhere else would move the estimation boundary in one cell and not the others.

## 3. The data contract it consumes, and the as-of rule

Long-only, fully invested, a 35% per-sleeve cap, a 1% absolute no-trade band and a 5% one-way turnover cap **on traded notional**. The band is applied to the target and the residual it leaves is brought back inside the constraint set before the book is traded, because a banded book can otherwise sum to something other than one. Cost is `2 x one-way turnover x per-side rate`: the rate is charged on traded notional, and one-way turnover is half of it. The no-mean cell bounds the weight vector's norm, so that nothing but the constraint set decides its answer.

## 4. The worked example on small numbers, with the identity checked

Minimum variance on a diagonal covariance has a closed form, so the worked example checks the solver against the reciprocal variances and then shows the cap binding; the constraint set is checked separately by the projection that clipping alone would get wrong.

The cell below runs on numbers small enough to check by hand and asserts the identity, so a reader can see the arithmetic rather than take the module's word for it.

In [1]:
import numpy as np
import pandas as pd

from portfolio_workbench.construct import constraints, families

covariance = pd.DataFrame(np.diag([0.04, 0.01, 0.0025]), index=list("abc"), columns=list("abc"))

# The closed form: weights proportional to the reciprocal variances, which the solver reproduces
# when nothing binds.
loose = families.minimum_variance(covariance, cap=1.0)
expected = np.array([1 / 0.04, 1 / 0.01, 1 / 0.0025])
expected = expected / expected.sum()
assert np.allclose(loose.to_numpy(), expected, atol=1e-6)
print("uncapped closed form:", loose.round(4).to_dict())

# The mandate's 35% cap binds here, and the delivered book is the capped projection of it.
capped = families.minimum_variance(covariance)
assert abs(capped.max() - 0.35) < 1e-9, capped
print("capped book:", capped.round(4).to_dict())

# Clipping alone leaves the book short of one; the projection hands the removed weight back.
assert np.allclose(constraints.bounded_simplex(np.array([0.9, 0.05, 0.05]), cap=0.5), [0.5, 0.25, 0.25])

uncapped closed form: {'a': 0.0476, 'b': 0.1905, 'c': 0.7619}
capped book: {'a': 0.3, 'b': 0.35, 'c': 0.35}


## 5. The real run: inputs, parameters, provenance block

The provenance block is printed first, then the parameters this module decides under, then the entry point's own report. The report is the module's output rather than a transcription of it, so a number quoted from a notebook is the number the module prints.

In [2]:
from portfolio_workbench.data import loader, universe

document = loader.load_panel()
months = document.months
print(f"snapshot {document.snapshot_id}, taken as of {document.as_of}")
print(f"panel {len(months)} months {months.min()}..{months.max()} across {len(universe.TICKERS)} sleeves")
print("manifest fields: " + ", ".join(sorted(document.manifest)))

from portfolio_workbench.construct import constraints, families, means

print(f"cap {constraints.CAP:.0%}, no-trade band {constraints.BAND:.0%}, turnover cap {constraints.TURNOVER_CAP:.0%}, "
      f"cost {constraints.COST_BP:.0f} bp per side on traded notional")
from portfolio_workbench.compare import registry

print("mean-input cells: " + ", ".join(cell["mean"] or "none" for cell in registry.STAGE_C))
print(f"risk aversion: gamma {families.RISK_AVERSION}")

snapshot 2026-09-13, taken as of 2026-09-13T07:56:28+00:00
panel 191 months 2010-09..2026-07 across 11 sleeves
manifest fields: created, excluded, files, instruments, snapshot_id, window
cap 35%, no-trade band 1%, turnover cap 5%, cost 10 bp per side on traded notional
mean-input cells: sample, black_litterman, none
risk aversion: gamma 0.3333333333333333


In [3]:
import subprocess
import sys
from pathlib import Path

# The package is imported from the repository root, so the run needs the root as its working
# directory rather than wherever the kernel was started. Walked up from the kernel's own directory
# rather than written in at generation time: an absolute path here would name one workstation, and
# the notebook is a file every reader runs on their own.
root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "portfolio_workbench").is_dir()
)

finished = subprocess.run(
    [sys.executable, "-m", "portfolio_workbench.construct.families"], capture_output=True, text=True, cwd=root
)
print(finished.stdout)
assert finished.returncode == 0, finished.stderr

## 6. Results, and how to read them, including the resolution limit and what a reader must not conclude

Weight concentration is the resolution limit here: at eleven sleeves a 35% cap binds often, and a cell whose book sits on the bound is a result about the constraint set rather than about its objective. The entry point prints how often the cap bound on each book and how much of each target was left untaken at the turnover cap for that reason. A reader must not read a tight weight vector as a confident one: minimum variance has no mean in it and is exposed to covariance error, while the mean-variance cells are exposed to mean error instead, and both are visible in the weight stability the table prints.

## 7. What this module does not establish

Nothing here establishes that an objective is the right one for a mandate, or that a better-constrained book will perform better. The no-mean cell establishes only that the constraint set is the estimator in that case. Nothing here addresses implementation shortfall, market impact or the size of the trade relative to the sleeve's turnover, none of which this panel observes.